### Results

Predict and Judge on test sets

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os

from dotenv import load_dotenv
import dspy

load_dotenv()

openai_key = os.getenv(
        "OPENAI_API_KEY"
    )


/Users/catherine/Library/Caches/pypoetry/virtualenvs/afan-WLhS1US3-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Predict: gpt-4o-mini

In [4]:
prediction_lm = dspy.LM("openai/gpt-4o-mini", api_key=openai_key)

dspy.configure(lm=prediction_lm)

## Heroes

In [5]:
from afan.dataset import load_and_preprocess_dataset

hero_test = load_and_preprocess_dataset(
    dataset_name="hero_test",
    data_dir="../data/test/"
)
hero_test.shape[0]

76

### Baseline: base_hero_predict

predictor: Predict

signature: BasicHeroSignature

In [12]:
from afan.prompts.signatures import BasicHeroSignature
from afan.utils import predict

base_hero_predict = predict(
    df=hero_test,
    predictor=dspy.Predict(BasicHeroSignature),
    entity="entity"
)

base_hero_predict.shape[0]

76

In [16]:
base_hero_predict.head()

,ID,text,entities,predicted_entity
0,804,Exxon Mobil Lends Its Support to a Carbon Tax ...,"[Carbon tax, The group]",carbon tax proposal
1,366,Mongolians sip 'oxygen cocktails' to cope with...,"[NGOs and Non-profit organisations, Mongolian ...",air purifiers
2,722,America Is Doubling Down On Climate Progress T...,"[American state governments, cities, state gov...",Sierra Club
3,310,The US produced more energy from renewable sou...,"[the US, Renewables]",renewable energy sources
4,869,2 protesters arrested after laying down on tra...,[Washington Governor Jay Inslee D],Governor Jay Inslee


### CoT BasicHeroSignature

base_hero_cot

In [13]:
base_hero_cot = predict(
    df=hero_test,
    predictor=dspy.ChainOfThought(BasicHeroSignature),
    entity="entity"
)

base_hero_cot.shape[0]

76

### Predict NarrativeArc

narrative_hero_predict

In [14]:
from afan.prompts.signatures import NarrativeArcSignature

narrative_hero_predict = predict(
    df=hero_test,
    predictor=dspy.Predict(NarrativeArcSignature),
    entity="hero"
)

narrative_hero_predict.shape[0]

76

### CoT NarrativeArc

In [15]:
narrative_hero_cot = predict(
    df=hero_test,
    predictor=dspy.ChainOfThought(NarrativeArcSignature),
    entity="hero"
)

narrative_hero_cot.shape[0]

76

## Villains

In [19]:
# remove dataset so we don't use it by mistake

del hero_test

In [21]:
villain_test = load_and_preprocess_dataset(
    dataset_name="villain_test",
    data_dir="../data/test/"
)

villain_test.shape[0]

102

### Predict NarrativeArc

In [26]:
narrative_villain_predict = predict(
    df=villain_test,
    predictor=dspy.Predict(NarrativeArcSignature),
    entity="villain"
)

In [27]:
narrative_villain_cot = predict(
    df=villain_test,
    predictor=dspy.ChainOfThought(NarrativeArcSignature),
    entity="villain"
)

## Victims

In [28]:
del villain_test

In [29]:
victim_test = load_and_preprocess_dataset(
    dataset_name="victim_test",
    data_dir="../data/test/"
)

victim_test.shape[0]

84

### Predict NarrativeArc

In [30]:
narrative_victim_predict = predict(
    df=victim_test,
    predictor=dspy.Predict(NarrativeArcSignature),
    entity="victim"
)
narrative_victim_predict.shape[0]

84

### CoT NarrativeArc

In [31]:
narrative_victim_cot = predict(
    df=victim_test,
    predictor=dspy.ChainOfThought(NarrativeArcSignature),
    entity="victim"
)

# Judge

In [32]:
judge_lm = dspy.LM("gpt-4.1-mini", api_key=openai_key)

dspy.configure(lm=judge_lm)

In [34]:
from afan.prompts.judges import EntitiesMatchFewShot
from afan.utils import judge

### Judge Heroes

#### judge_base_hero_predict

In [35]:
judge_base_hero_predict = judge(
    df=base_hero_predict,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 57.89%


In [37]:
from afan.dataset import save_tsv

save_tsv(
    df=judge_base_hero_predict,
    data_dir="results",
    name="judge_base_hero_predict"
)

#### judge_base_hero_cot

In [38]:
judge_base_hero_cot = judge(
    df=base_hero_cot,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 56.58%


In [39]:
save_tsv(
    df=judge_base_hero_cot,
    data_dir="results",
    name="judge_base_hero_cot"
)

#### judge narrative_hero_predict

In [40]:
judge_narrative_hero_predict = judge(
    df=narrative_hero_predict,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 64.47%


In [41]:
save_tsv(
    df=judge_narrative_hero_predict,
    data_dir="results",
    name="judge_narrative_hero_predict"
)

#### judge narrative_hero_cot

In [42]:
judge_narrative_hero_cot = judge(
    df=narrative_hero_cot,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 63.16%


In [43]:
save_tsv(
    df=judge_narrative_hero_cot,
    data_dir="results",
    name="judge_narrative_hero_cot"
)

### Judge Villains

In [44]:
judge_narrative_villain_predict = judge(
    df=narrative_villain_predict,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 66.67%


In [45]:
save_tsv(
    df=judge_narrative_villain_predict,
    data_dir="results",
    name="judge_narrative_villain_predict"
)

In [46]:
judge_narrative_villain_cot = judge(
    df=narrative_villain_cot,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 61.76%


In [47]:
save_tsv(
    df=judge_narrative_villain_cot,
    data_dir="results",
    name="judge_narrative_villain_cot"
)

### Judge Victims

In [48]:
judge_narrative_victim_predict = judge(
    df= narrative_victim_predict,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 60.71%


In [49]:
save_tsv(
    df=judge_narrative_victim_predict,
    data_dir="results",
    name="judge_narrative_victim_predict"
)

In [50]:
judge_narrative_victim_cot = judge(
    df=narrative_victim_cot,
    judge_match=dspy.Predict(EntitiesMatchFewShot)
)

Accuracy: 58.33%


In [51]:
save_tsv(
    df=judge_narrative_victim_cot,
    data_dir="results",
    name="judge_narrative_victim_cot"
)